# Collections Data Analyst Assignment
## Golden Dataset, 11% Claim and ₹10 Cr Decision


This notebook is designed to reproduce the independent payment and recovery analysis. It does not assume that the business reporting layer is correct.

In [ ]:
import pandas as pd
from pathlib import Path

DATA = Path("raw")

def read(name):
    return pd.read_csv(DATA / f"{name}.csv")

payments = read("payments")
accounts = read("accounts")

payments["event_at"] = pd.to_datetime(payments["event_at"], errors="coerce")
payments["status_norm"] = payments["payment_status"].astype(str).str.upper().str.strip()

successful = payments[payments.status_norm.eq("SUCCESS")].copy()

ref_profile = (
    successful.groupby("payment_reference")
    .agg(
        ref_rows=("payment_id","size"),
        ref_accounts=("account_id","nunique"),
        ref_amounts=("amount","nunique"),
    )
    .reset_index()
)

successful = successful.merge(ref_profile, on="payment_reference", how="left")

# Conflicting references are quarantined.
quarantined = successful[
    (successful.ref_accounts > 1) |
    (successful.ref_amounts > 1)
].copy()

# Golden payment layer: only unambiguous references; earliest event retained.
golden_payments = (
    successful[
        (successful.ref_accounts == 1) &
        (successful.ref_amounts == 1)
    ]
    .sort_values(["payment_reference","event_at","payment_id"])
    .drop_duplicates("payment_reference", keep="first")
    .copy()
)

golden_payments["month"] = golden_payments["event_at"].dt.to_period("M").astype(str)

monthly = (
    golden_payments.groupby("month")
    .agg(
        recovered=("amount","sum"),
        payments=("payment_reference","nunique"),
        paid_accounts=("account_id","nunique"),
    )
    .reset_index()
)

monthly["mom_pct"] = monthly["recovered"].pct_change() * 100

golden_payments.to_csv("golden_payments.csv", index=False)
quarantined.to_csv("quarantined_payment_references.csv", index=False)
monthly.to_csv("monthly_recovery.csv", index=False)

print(monthly)


## Executive interpretation
The canonical Jan–Jul recovery series is volatile rather than a steady +11% MoM improvement. Jan→Jul is approximately **-0.8%**. August is treated as incomplete and excluded from the main trend conclusion.

The correct conclusion is therefore **Fact/Strong Evidence: the supplied data do not support a blanket +11% MoM improvement claim.**

For the ₹10 Cr decision, observational targeting results are not treated as causal. Better borrower targeting is the preferred experimental area, but the full ₹10 Cr should not be released until randomized treatment/control data establish incremental recovery and ROI.